In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Configuration ---
INPUT_FILE = 'extracted_features/hg38_100kb_filtered_bins/EE87920.hg38.frag.tsv_BIN.csv'
OUTPUT_IMAGE_FILE = 'cfdna_feature_analysis.png'
BINS_TO_ANALYZE = 500

def analyze_and_plot_features(file_path):
    """
    Reads cfDNA feature data, calculates correlations, and generates plots.

    Args:
        file_path (str): The path to the input CSV file.
    """
    print(f"Reading data from '{file_path}'...")
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: Input file not found at '{file_path}'.")
        return

    # 1. Subset the data to the first 500 rows (bins)
    print(f"Analyzing the first {BINS_TO_ANALYZE} bins...")
    df_subset = df.head(BINS_TO_ANALYZE).copy()

    # 2. Isolate numeric feature columns for correlation analysis
    # We assume 'chrom', 'start', 'end' are identifiers and exclude them.
    features_df = df_subset.select_dtypes(include=['number'])
    # If 'start' and 'end' are numeric but not features, drop them explicitly
    if 'start' in features_df.columns:
        features_df = features_df.drop(columns=['start', 'end'])

    print("Calculating pairwise correlation matrix...")
    correlation_matrix = features_df.corr()

    # 3. Extract specific correlations for the line plot legend
    try:
        corr_fslr_mds = correlation_matrix.loc['fslr', 'mds']
        corr_fslr_rel = correlation_matrix.loc['fslr', 'relative_read_counts']
        corr_mds_rel = correlation_matrix.loc['mds', 'relative_read_counts']

        # Create descriptive labels for the legend
        fslr_label = f"fslr (corr w/ mds: {corr_fslr_mds:.2f}, w/ rel_counts: {corr_fslr_rel:.2f})"
        mds_label = f"mds (corr w/ fslr: {corr_fslr_mds:.2f}, w/ rel_counts: {corr_mds_rel:.2f})"
        rel_counts_label = f"relative_read_counts (corr w/ fslr: {corr_fslr_rel:.2f}, w/ mds: {corr_mds_rel:.2f})"
    except KeyError as e:
        print(f"Error: A required column ({e}) was not found for plotting. Please check the CSV file.")
        return
        
    # 4. Create the plots
    print("Generating plots...")
    fig, axes = plt.subplots(2, 1, figsize=(18, 22), gridspec_kw={'height_ratios': [2, 1]})
    fig.suptitle('Analysis of cfDNA Fragmentomic Features (First 500 Bins)', fontsize=20)

    # --- Plot 1: Correlation Heatmap ---
    sns.heatmap(
        correlation_matrix,
        annot=True,
        cmap='coolwarm',
        fmt='.2f',
        linewidths=.5,
        ax=axes[0]
    )
    axes[0].set_title('Pairwise Feature Correlation Heatmap', fontsize=16)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)


    # --- Plot 2: Line Plot of Selected Features ---
    df_subset.plot(y='fslr', ax=axes[1], label=fslr_label, style='-')
    df_subset.plot(y='mds', ax=axes[1], label=mds_label, style='--')
    df_subset.plot(y='relative_read_counts', ax=axes[1], label=rel_counts_label, style=':')

    axes[1].set_title('Feature Values Across Genomic Bins', fontsize=16)
    axes[1].set_xlabel('Genomic Bin Index (100kb bins)', fontsize=12)
    axes[1].set_ylabel('Feature Value', fontsize=12)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(title="Features & Pairwise Correlations", fontsize=10)
    axes[1].set_xlim(0, BINS_TO_ANALYZE) # Ensure x-axis is neat

    # 5. Save the combined figure
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make room for suptitle
    plt.savefig(OUTPUT_IMAGE_FILE, dpi=300)
    
    print(f"\nAnalysis complete. Plots saved to '{OUTPUT_IMAGE_FILE}'.")


if __name__ == '__main__':
    analyze_and_plot_features(INPUT_FILE)